# Compare 2010 and 2020: differential privacy evidence in pyncoda

## Overview
The 2020 Census injects noise into block-level population counts to protect privacy
(the TopDown Algorithm); the 2010 Census did not. This notebook compares the housing
unit and person record products for one community across both vintages and shows, step
by step, where the noise appears, what it costs, and what it leaves untouched:

1. County totals by vintage (what the noise does NOT change)
2. Block-level agreement between the person and housing tables (the noise signature)
3. Person-to-housing-unit assignment rates (the noise cost)
4. The displacement metric (how far assignments can move, in meters)
5. Comparison heat maps (whether subpopulation concentrations survive the noise)

## Required inputs
Run the HUA + PREC + Intersect workflow (`ncoda_07kv7_HUA_PREC_Disability.ipynb`) for
BOTH vintages of the community first; this notebook only reads its saved products, and
needs no Census API key.

## References
U.S. Census Bureau (2023). "Disclosure Avoidance and the 2020 Census: How the TopDown
Algorithm Works." C2020BR-04. Invariants: state total population, block-level housing
unit counts, block-level occupied group quarters facility counts. "All other population
and housing characteristics, including population counts for every geography below the
state level, have had noise introduced."

U.S. Census Bureau (2023). "Disclosure Avoidance and the 2020 Census Redistricting
Data." C2020BR-02. Guidance: "DON'T use data for individual blocks. Instead, aggregate
data into larger areas."


## Description of Program
- program:    ncoda_08cv1_Compare2010_2020
- task:       compare HUI and PREC results across the 2010 and 2020 vintages
- Current Version: v1
- 2026-09-16 - initial version (issue #144)
- project:    Interdependent Networked Community Resilience Modeling Environment (IN-CORE), Subtask 5.2 - Social Institutions
- author:     Nathanael Rosenheim and Emmanuel Randle


In [ ]:
# To reload submodules need to use this magic command to set autoreload on
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

from pyncoda.ncoda_00g_community_options import (
    communities_dictionary, get_community_id_by_name, list_community_options)
from pyncoda.ncoda_00e_geoutilities import (
    load_block_points, assignment_displacement_m)
from pyncoda.ncoda_04d_heatmaps import (
    compare_heatmaps, folium_comparison_map)


In [ ]:
# Select the community. Both vintages of its products must exist on disk.
community_id_by_name = 'Grays Harbor, WA: NSI Building inventory for Grays Harbor County, WA'
community_id, focalplace, countyname, countyfips = get_community_id_by_name(community_id_by_name)

seed = 9876
vintages = ['2010', '2020']
outputfolder = 'OutputData'
state_county = countyfips

# The products this notebook reads, per vintage. hui_linkage carries the
# housing inventory with householder characteristics and block geography;
# prechui_bldg carries linked persons with building coordinates.
paths = {}
for v in vintages:
    base = f'{outputfolder}/{community_id}'
    paths[v] = {
        'hui': f'{base}/hui_linkage_v2-2-0_{state_county}_{v}_rs{seed}.csv',
        'prec': f'{base}/prechui_bldg_v2-1-0_{state_county}_{v}_rs{seed}.csv',
        'coreprec': f'{base}/03_BaseInventory/CorePREC_{v}_{state_county}_{v}.csv',
        'tiger': (f'{base}/01_CommunitySourceData/'
                  f'tl_{v}_{state_county}_tabblockplacepuma{v[2:]}EPSG4269.csv'),
    }
    for name, p in paths[v].items():
        if not os.path.exists(p):
            raise FileNotFoundError(
                f'{v} {name} product missing: {p} - run the HUA + PREC + '
                f'Intersect workflow for basevintage {v} first.')

data = {v: {'hui': pd.read_csv(paths[v]['hui'], low_memory=False),
            'prec': pd.read_csv(paths[v]['prec'], low_memory=False),
            'coreprec': pd.read_csv(paths[v]['coreprec'], low_memory=False)}
        for v in vintages}
print(f'{countyname}: products loaded for {vintages}')


## 1. County totals by vintage

State total population is a TopDown invariant, and county aggregation cancels most of
the block noise, so these totals should match the published Census counts exactly in
both vintages. This is the first thing the noise does NOT change.


In [ ]:
def unassigned_mask(huid_series):
    return huid_series.isna() | (huid_series.astype(str) == '-999')

totals_rows = []
for v in vintages:
    prec, hui = data[v]['prec'], data[v]['hui']
    assigned = ~unassigned_mask(prec['huid'])
    totals_rows.append({
        'vintage': v,
        'person records': len(prec),
        'housing units': len(hui),
        'group quarters residents': int((prec['gqtype'].fillna(0) > 0).sum()),
        'persons assigned a huid': int(assigned.sum()),
        'percent assigned': round(assigned.mean() * 100, 2),
    })
totals_df = pd.DataFrame(totals_rows).set_index('vintage')
display(totals_df)


## 2. Block-level agreement between the person and housing tables

The person products are built from the Census P tables, the housing products from the
H tables. Before any linkage runs, do those two table families agree about how many
people each block holds? In 2010 they nearly always do, and the few disagreements lean
one way only (households larger than the reporting cap of 7). In 2020 the injected
noise makes them disagree in BOTH directions on most blocks - the noise signature.


In [ ]:
agreement_rows = []
agreement_stats = {}
for v in vintages:
    coreprec, hui = data[v]['coreprec'], data[v]['hui']
    p_col = [c for c in coreprec.columns if c.startswith(f'Block{v}')][0]
    h_col = f'Block{v}str' if f'Block{v}str' in hui.columns else f'Block{v}'
    p_side = coreprec.assign(
        blk=coreprec[p_col].astype(str).str.extract(r'(\d{15})', expand=False)
    ).groupby('blk').size()
    h_side = hui.assign(
        blk=hui[h_col].astype(str).str.extract(r'(\d{15})', expand=False)
    ).groupby('blk')['numprec'].sum()
    both = pd.DataFrame({'p': p_side, 'h': h_side}).fillna(0).astype(int)
    both = both[(both.p > 0) | (both.h > 0)]
    diff = both.p - both.h
    agreement_stats[v] = {'agree_pct': round((diff == 0).mean() * 100, 1),
                          'blocks_p_over_h': int((diff > 0).sum()),
                          'blocks_h_over_p': int((diff < 0).sum())}
    agreement_rows.append({
        'vintage': v, 'populated blocks': len(both),
        'blocks agreeing exactly': int((diff == 0).sum()),
        'percent agreeing': agreement_stats[v]['agree_pct'],
        'blocks persons > capacity': agreement_stats[v]['blocks_p_over_h'],
        'person surplus in those blocks': int(diff[diff > 0].sum()),
        'blocks capacity > persons': agreement_stats[v]['blocks_h_over_p'],
    })
agreement_df = pd.DataFrame(agreement_rows).set_index('vintage')
display(agreement_df)
print('Reading: one-directional disagreement (2010) is the size-7 reporting cap;')
print('symmetric two-directional disagreement (2020) is injected noise.')


## 3. What the noise costs: assignment rates

The linkage places every person the arithmetic permits, block by block. Where the
person tables say a block holds more people than the housing tables can seat, the
surplus is unplaceable within the block. In 2010 that surplus is only the size-cap
effect, which the linkage numprec adjustment absorbs: assignment reaches 100 percent.
In 2020 the noise surplus remains: the unassigned share below IS the block-level noise,
not a defect in the merge.


In [ ]:
display(totals_df[['persons assigned a huid', 'percent assigned']])


## 4. The displacement metric

When the merge is allowed to look beyond the block (the geography ladder), how far do
people move? The metric: distance between a person's own block and the block of their
assigned housing unit, using Census block internal points. Under the default
block-only configuration every placed person stays home, so every distance is zero -
the point of this cell is the machinery, which the geography-widening experiments in
issue #140 used to price every configuration in meters.


In [ ]:
displacement_rows = []
for v in vintages:
    block_points = load_block_points(paths[v]['tiger'], v)
    displacement = assignment_displacement_m(data[v]['prec'], block_points, v).dropna()
    displacement_rows.append({
        'vintage': v, 'persons with a distance': len(displacement),
        'mean (m)': round(float(displacement.mean()), 1),
        'share at zero (own block)': f'{(displacement == 0).mean():.2%}',
        'max (m)': round(float(displacement.max()), 1),
    })
display(pd.DataFrame(displacement_rows).set_index('vintage'))


## 5. Do subpopulation concentrations survive the noise?

Heat maps of the same subpopulation in 2010 and 2020, on one shared grid, with two
honest numbers: the correlation of the density surfaces and the overlap of the
top-decile hotspot cells. Then the vulnerability contrast within 2020: a
high-vulnerability group against a low-vulnerability group.


In [ ]:
def person_unit_frame(v):
    """Assigned household persons joined to their unit tenure and income."""
    prec, hui = data[v]['prec'], data[v]['hui']
    income_col = 'randincomeB19101'
    unit_cols = ['huid', 'ownershp', income_col]
    frame = prec[~unassigned_mask(prec['huid'])].merge(
        hui[unit_cols], on='huid', how='left', validate='m:1')
    frame = frame[frame['gqtype'].fillna(0) == 0]
    occupied = hui[hui['ownershp'].isin([1, 2])]
    quartiles = occupied[income_col].quantile([0.25, 0.75])
    return frame, float(quartiles[0.25]), float(quartiles[0.75])

def town_labels(v):
    prec = data[v]['prec']
    place_col = f'placeNAME{v[2:]}'
    towns = (prec.dropna(subset=[place_col])
             .loc[lambda d: ~d[place_col].str.contains('nincorporated', na=False)]
             .groupby(place_col)
             .agg(x=('x', 'median'), y=('y', 'median'), n=('x', 'size'))
             .sort_values('n', ascending=False).head(6)
             .reset_index().rename(columns={place_col: 'name'}))
    return towns

subpop = {}
for v in vintages:
    frame, p25, p75 = person_unit_frame(v)
    subpop[v] = {
        'low_income_renters_65plus': frame[
            (frame['randagePCT12'] >= 65) & (frame['ownershp'] == 2)
            & (frame['randincomeB19101'] <= p25)],
        'high_vulnerability': frame[
            (frame['randincomeB19101'] <= p25) & (frame['randagePCT12'] >= 65)
            & ((frame['disability'] == 1) | (frame['numprec'] == 1))],
        'low_vulnerability': frame[
            (frame['ownershp'] == 1) & (frame['family'] == 1)
            & (frame['randincomeB19101'] >= p75) & (frame['disability'] == 0)],
    }
    print(f"{v}: renters65+ {len(subpop[v]['low_income_renters_65plus']):,} | "
          f"high-vuln {len(subpop[v]['high_vulnerability']):,} | "
          f"low-vuln {len(subpop[v]['low_vulnerability']):,} "
          f"(income quartiles ${p25:,.0f} / ${p75:,.0f})")


In [ ]:
# The vintage comparison: the same subpopulation, 2010 against 2020.
figures_folder = f'{outputfolder}/{community_id}'
vintage_stats = compare_heatmaps(
    subpop['2010']['low_income_renters_65plus'],
    subpop['2020']['low_income_renters_65plus'],
    '2010', '2020',
    f'Low-income renters age 65+, {countyname}',
    f'{figures_folder}/compare_renters65_2010_2020.png',
    place_labels=town_labels('2020'))
print(vintage_stats)


In [ ]:
# The vulnerability contrast within 2020.
contrast_stats = compare_heatmaps(
    subpop['2020']['high_vulnerability'],
    subpop['2020']['low_vulnerability'],
    'High vulnerability: low income, 65+, disability or living alone',
    'Low vulnerability: high income homeowners, family, no disability',
    f'Vulnerability contrast, {countyname} 2020 (rs{seed})',
    f'{figures_folder}/compare_vulnerability_2020.png',
    place_labels=town_labels('2020'))
print(contrast_stats)

folium_comparison_map(
    subpop['2020']['high_vulnerability'], subpop['2020']['low_vulnerability'],
    'High vulnerability', 'Low vulnerability',
    f'{figures_folder}/compare_vulnerability_2020.html')
print('interactive map saved')


## Reading the evidence

- **County totals are exact in both vintages** - the noise is designed to cancel in
  aggregates, and it does.
- **Block-level disagreement is the noise**: near-perfect one-directional agreement in
  2010, symmetric disagreement on most blocks in 2020, present in the published tables
  before pyncoda touches them.
- **The unassigned share in 2020 equals the per-block person surplus exactly** - a
  capacity floor, not a merge defect. Reaching 100 percent requires widening the merge
  geography, priced in meters by the displacement metric (issue #144 and the #140
  configuration table).
- **Concentrations survive**: subpopulation hot spots stay in the same places across
  vintages despite the noise, consistent with the Census Bureau guidance to aggregate
  rather than read single blocks.
- For uncertainty in any final statistic, run the seed ensemble: aggregates hold to
  the person across seeds while individual placements vary.
